# Módulo 10 · Mejoras de calidad

Cada ajuste se valida con el conjunto de evaluación del módulo 09. Se utiliza la
recuperación híbrida con el modelo E5 como referencia.

In [1]:
from pathlib import Path

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
DATA_DIR = ROOT / "data" / "novatech"

In [2]:
from fundamenta.chunking import chunk_documents
from fundamenta.embeddings import SentenceTransformerEmbedder
from fundamenta.evaluation import evaluate_retrieval, load_questions
from fundamenta.loaders import load_directory
from fundamenta.retriever import Retriever

documents = load_directory(DATA_DIR)
questions = load_questions(ROOT / "evaluacion" / "preguntas.jsonl")
embedder = SentenceTransformerEmbedder()


def measure(retriever: Retriever, k: int = 4) -> str:
    report = evaluate_retrieval(lambda q, n: retriever.search(q, k=n), questions, k=k)
    return f"acierto@{k} {report.hit_rate:4.0%}   MRR {report.mrr:.3f}"


base_chunks = chunk_documents(documents)

## 1. Contexto de sección en el índice

In [3]:
for contextual in (False, True):
    for mode in ("lexical", "hybrid"):
        retriever = Retriever(base_chunks, embedder=embedder, mode=mode, contextual=contextual)
        print(f"contextual={contextual!s:5} {mode:8} {measure(retriever)}")

contextual=False lexical  acierto@4  78%   MRR 0.741


contextual=False hybrid   acierto@4  89%   MRR 0.846


contextual=True  lexical  acierto@4  78%   MRR 0.759


contextual=True  hybrid   acierto@4  93%   MRR 0.824


## 2. Peso de la búsqueda léxica en la fusión

In [4]:
retriever = Retriever(base_chunks, embedder=embedder, mode="hybrid")
for weight in (1.0, 0.7, 0.5, 0.3):
    retriever.lexical_weight = weight
    print(f"peso léxico {weight:.1f}   {measure(retriever)}")
retriever.mode = "semantic"
print(f"solo semántica   {measure(retriever)}")

peso léxico 1.0   acierto@4  89%   MRR 0.812


peso léxico 0.7   acierto@4  93%   MRR 0.821


peso léxico 0.5   acierto@4  93%   MRR 0.824


peso léxico 0.3   acierto@4  93%   MRR 0.806


solo semántica   acierto@4  96%   MRR 0.824


## 3. Tamaño de fragmento

In [5]:
for size in (300, 500, 800):
    chunks = chunk_documents(documents, chunk_size=size, overlap=size // 10)
    retriever = Retriever(chunks, embedder=embedder, mode="hybrid")
    print(f"chunk_size {size:>4} ({len(chunks):>2} fragmentos)   {measure(retriever)}")

chunk_size  300 (61 fragmentos)   acierto@4  93%   MRR 0.818


chunk_size  500 (41 fragmentos)   acierto@4  93%   MRR 0.824


chunk_size  800 (32 fragmentos)   acierto@4  93%   MRR 0.824


## 4. Número de fragmentos recuperados (k)

Más fragmentos aumentan la probabilidad de incluir la respuesta, a cambio de un prompt
más largo y costoso.

In [6]:
retriever = Retriever(base_chunks, embedder=embedder, mode="hybrid")
for k in (1, 2, 4, 6, 8):
    print(f"k={k}   {measure(retriever, k=k)}")

k=1   acierto@1  78%   MRR 0.778


k=2   acierto@2  78%   MRR 0.778


k=4   acierto@4  93%   MRR 0.824


k=6   acierto@6  93%   MRR 0.824


k=8   acierto@8  96%   MRR 0.829


## 5. Reescritura de consultas

Con `PipelineConfig(rewrite_queries=True)` un LLM reformula la pregunta antes de
buscar. El siguiente ejemplo muestra el efecto con una reformulación escrita a mano,
del tipo que produciría el modelo:

In [7]:
original = "¿Ayuda la empresa con los gastos de deporte?"
rewritten = "reembolso cuota gimnasio actividad deportiva bienestar beneficios"

for query in (original, rewritten):
    top = retriever.search(query, k=1)[0].document
    print(f"{query}\n  -> {top.source} | {top.text[:70].replace(chr(10), ' ')}…\n")

¿Ayuda la empresa con los gastos de deporte?
  -> politica-de-gastos-y-viajes.md | ## Principios generales  NovaTech cubre los gastos razonables en los q…

reembolso cuota gimnasio actividad deportiva bienestar beneficios
  -> beneficios-2026.pdf | 4. Bienestar NovaTech reembolsa hasta 30 € al mes de la cuota de gimna…

